In [1]:
import torch
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm
import os

In [2]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)


device: mps


In [ ]:
from tablevault import tablevault

vault = tablevault.Vault(user_id="jinjin",
                            process_name="direct_nli_entailment_mrpc",
                            arango_url="http://localhost:8629",
                            arango_db="tv_experiment_1",
                            arango_username="tablevault_user",
                            arango_password="tablevault_password",
                            new_arango_db=False,               
                            arango_root_username="root",
                            arango_root_password="passwd",
                            description_embedding_size=3072,
                        )

from openai import OpenAI


openai_key_file = "/Users/jinjinzhao/Documents/work_projects/my_keys/my_keys/openai_jinjin.key"
with open(openai_key_file, 'r') as f:
    openai_key = f.read()

os.environ["OPENAI_API_KEY"] = openai_key

client = OpenAI()

In [ ]:
def get_embeddings(text):
    return client.embeddings.create(
            input=text,
            model="text-embedding-3-large"
        ).data[0].embedding

In [3]:
model_name = "typeform/distilbert-base-uncased-mnli"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
model.eval()

id2label = {int(k): v for k, v in model.config.id2label.items()} if isinstance(next(iter(model.config.id2label.keys())), str) else dict(model.config.id2label)
label2id_norm = {str(v).lower(): int(k) for k, v in id2label.items()}

def find_label_id(candidates, mapping):
    for cand in candidates:
        if cand in mapping:
            return mapping[cand]
    raise ValueError(f"Could not find any of {candidates} in labels: {mapping}")

entailment_id = find_label_id(["entailment"], label2id_norm)
contradiction_id = find_label_id(["contradiction", "contradictory"], label2id_norm)

print("model:", model_name)
print("id2label:", id2label)
print("entailment_id:", entailment_id)
print("contradiction_id:", contradiction_id)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

model: typeform/distilbert-base-uncased-mnli
id2label: {0: 'ENTAILMENT', 1: 'NEUTRAL', 2: 'CONTRADICTION'}
entailment_id: 0
contradiction_id: 2


In [4]:
ds = vault.query_item_content("glue_mrpc_validation")
ds = Dataset.from_dict(ds)
print(ds)
print(ds[0])

sent1 = ds["sentence1"]
sent2 = ds["sentence2"]
y_true = np.array(ds["label"])

print("num_examples:", len(y_true))
print("positive_rate:", float(y_true.mean()))


Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}
num_examples: 408
positive_rate: 0.6838235294117647


In [5]:
batch_size = 64
margin_12_all = []
margin_21_all = []
avg_margin_all = []
preds = []

with torch.no_grad():
    for i in tqdm(range(0, len(ds), batch_size)):
        batch_s1 = sent1[i:i + batch_size]
        batch_s2 = sent2[i:i + batch_size]

        enc_12 = tokenizer(
            batch_s1,
            batch_s2,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt",
        )
        enc_21 = tokenizer(
            batch_s2,
            batch_s1,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt",
        )

        enc_12 = {k: v.to(device) for k, v in enc_12.items()}
        enc_21 = {k: v.to(device) for k, v in enc_21.items()}

        logits_12 = model(**enc_12).logits
        logits_21 = model(**enc_21).logits

        margin_12 = (logits_12[:, entailment_id] - logits_12[:, contradiction_id]).detach().cpu().numpy()
        margin_21 = (logits_21[:, entailment_id] - logits_21[:, contradiction_id]).detach().cpu().numpy()
        avg_margin = (margin_12 + margin_21) / 2.0

        batch_preds = (avg_margin > 0.0).astype(int)

        margin_12_all.extend(margin_12.tolist())
        margin_21_all.extend(margin_21.tolist())
        avg_margin_all.extend(avg_margin.tolist())
        preds.extend(batch_preds.tolist())




  0%|          | 0/7 [00:00<?, ?it/s]

done


In [ ]:
margin_12_all = np.array(margin_12_all)
margin_21_all = np.array(margin_21_all)
avg_margin_all = np.array(avg_margin_all)
y_pred = np.array(preds)

print("done")


vault.create_record_list("distilbert-margin-and-prediction", column_names=["prediction", "margin_12" , "margin_21"])

for i in range(len(y_pred)):
    vault.append_record("distilbert-margin-and-prediction", 
                        {
                            "prediction": y_pred[i],
                            "margin_12": float(margin_12_all[i]),
                            "margin_21": float(margin_21_all[i]),
                        },
                       input_items = {
                           "glue_mrpc_validation": [i, i + 1],
                       }
                       )

description = "INSERT TEXT HERE ABOUT distilbert-margin-and-prediction"
embedding = get_embeddings(description)
vault.create_description("distilbert-margin-and-prediction", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("distilbert-margin-and-prediction", cat, embedding, prop)

In [6]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
report = classification_report(y_true, y_pred, target_names=['not_paraphrase', 'paraphrase'])

print({"accuracy": acc, "f1": f1})
print(classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"]))


{'accuracy': 0.7034313725490197, 'f1': 0.7729831144465291}
                precision    recall  f1-score   support

not_paraphrase       0.53      0.63      0.57       129
    paraphrase       0.81      0.74      0.77       279

      accuracy                           0.70       408
     macro avg       0.67      0.68      0.67       408
  weighted avg       0.72      0.70      0.71       408



In [7]:
for i in range(5):
    print("=" * 80)
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("margin_12:", float(margin_12_all[i]))
    print("margin_21:", float(margin_21_all[i]))
    print("avg_margin:", float(avg_margin_all[i]))
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))

mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

for i in mistakes:
    print("=" * 80)
    print("idx:", int(i))
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("margin_12:", float(margin_12_all[i]))
    print("margin_21:", float(margin_21_all[i]))
    print("avg_margin:", float(avg_margin_all[i]))
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))


sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
margin_12: 7.967843055725098
margin_21: 6.533724784851074
avg_margin: 7.250783920288086
true: 1 pred: 1
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
margin_12: -9.776514053344727
margin_21: -0.8201401233673096
avg_margin: -5.2983269691467285
true: 0 pred: 0
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
margin_12: -0.6154601573944092
margin_21: 6.547325134277344
avg

In [8]:
vault.create_record_list("direct_nli_entailment_mrpc_summary", column_names=["accuracy", "f1", "classification_report"])


summary = {
    "accuracy": float(acc),
    "f1": float(f1),
    "classification_report": str(report)
}

vault.append_record("direct_nli_entailment_mrpc_summary", summary,
                    input_items = {
                        "glue_mrpc_validation": [0, len(ds)],
                        "distilbert-margin-and-prediction": [0, len(ds)]
                    })

summary

description = "INSERT TEXT HERE ABOUT direct_nli_entailment_mrpc_summary"
embedding = get_embeddings(description)
vault.create_description("direct_nli_entailment_mrpc_summary", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("direct_nli_entailment_mrpc_summary", cat, embedding, prop)


{'dataset': 'glue/mrpc',
 'split': 'validation',
 'model': 'typeform/distilbert-base-uncased-mnli',
 'device': 'mps',
 'num_examples': 408,
 'accuracy': 0.7034313725490197,
 'f1': 0.7729831144465291}

In [ ]:
description = "INSERT TEXT HERE ABOUT direct_nli_entailment_mrpc process" # description of whole notebook
embedding = get_embeddings(description)
vault.create_description("direct_nli_entailment_mrpc", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. model: distilbert-base-uncased-MRPC

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("direct_nli_entailment_mrpc", cat, embedding, prop)